In [1]:
# build_one_to_many_graph.py

import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from sentence_transformers import SentenceTransformer
import os


c:\Users\Anshul\Desktop\Major Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ---------- CONFIG ----------
INPUT_FILE = "cleaned_pairs.csv"
OUTPUT_DIR = "graph_data_one_to_many"
EMBED_MODEL = "all-MiniLM-L6-v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)



In [4]:

# ----------------------------

# 1. Load cleaned dataset
df = pd.read_csv(INPUT_FILE)
print("Loaded dataset:", len(df))


Loaded dataset: 86376


In [5]:

# 2. Combine text
df["email_text"] = df["email_subject_clean"].fillna('') + " " + df["email_body_clean"].fillna('')
df["reply_text"] = df["reply_subject_clean"].fillna('') + " " + df["reply_body_clean"].fillna('')


In [8]:

# 3. Group by Email_ID (one email, many replies)
grouped = df.groupby("email_subject_clean" and "email_body_clean")

email_nodes = []
reply_nodes = []
edges = []

# track indices
email_id_to_node = {}
node_texts = []
node_types = []

# --- Create email nodes ---
for idx, (eid, group) in enumerate(grouped):
    email_text = group.iloc[0]["email_text"]  # same for all replies
    email_nodes.append(eid)
    email_id_to_node[eid] = len(node_texts)
    node_texts.append(email_text)
    node_types.append(0)  # email node

# --- Create reply nodes & edges ---
for eid, group in grouped:
    email_node_index = email_id_to_node[eid]
    for _, row in group.iterrows():
        reply_text = row["reply_text"]
        node_texts.append(reply_text)
        node_types.append(1)
        reply_node_index = len(node_texts) - 1
        edges.append((email_node_index, reply_node_index))  # email → reply


In [9]:

# 4. Encode embeddings
print("Encoding embeddings...")
model = SentenceTransformer(EMBED_MODEL)
embeddings = model.encode(node_texts, show_progress_bar=True, convert_to_numpy=True)
embeddings = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-9)


Encoding embeddings...


Batches: 100%|██████████| 2976/2976 [14:55<00:00,  3.32it/s]


In [10]:

# 5. Build edge_index tensor
edge_index = torch.tensor(np.array(edges).T, dtype=torch.long)
x = torch.tensor(embeddings, dtype=torch.float32)
node_type_tensor = torch.tensor(node_types, dtype=torch.long).unsqueeze(1)
x = torch.cat([x, node_type_tensor.float()], dim=1)


In [11]:

# 6. Create graph
data = Data(x=x, edge_index=edge_index)
print(data)
print("Graph created with", data.num_nodes, "nodes and", data.num_edges, "edges.")


Data(x=[95209, 385], edge_index=[2, 86376])
Graph created with 95209 nodes and 86376 edges.


In [12]:

# 7. Save
np.save(os.path.join(OUTPUT_DIR, "embeddings.npy"), embeddings)
np.save(os.path.join(OUTPUT_DIR, "edge_index.npy"), edge_index.numpy())
pd.DataFrame({"node_text": node_texts, "node_type": node_types}).to_csv(os.path.join(OUTPUT_DIR, "nodes.csv"), index=False)
print("✅ One-to-many graph saved.")


✅ One-to-many graph saved.
